# Подготовка данных для обучения моделей

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

__Выполнила Ларькова Дарья__

Материалы: 
* https://scikit-learn.org/stable/modules/compose.html#pipeline-chaining-estimators
* https://pytorch.org/docs/stable/data.html 
* https://pytorch.org/tutorials/beginner/data_loading_tutorial.html
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann


## Задачи для совместного разбора

1. Создайте синтетический датасет для задачи регрессии и представьте его в виде `torch.utils.data.Dataset`

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Считайте файл `bank-full.csv` ([источник](https://www.kaggle.com/datasets/hariharanpavan/bank-marketing-dataset-analysis-classification)) в виде `pd.DataFrame`.

Опишите класс `BankDatasetBase`. Решение должно удовлетворять следующим критериям:

* класс наследуется от `torch.utils.data.Dataset`;
* при создании объекта в конструктор передается набор данных в виде `pd.DataFrame`;
* объекты класса имеют поля `X` и `y` с признаками и метками соответственно;
* класс реализует интерфейс последовательностей (`__getitem__` + `__len__`);
* `obj[i]` возвращает кортеж, содержащий `i`-ую строку из `obj.X` (серию) и `i`-ую строку из `obj.y` (строку).
    
Создайте объект класса `BankDatasetBase` и продемонстрируйте работоспособность.

- [ ] Проверено на семинаре

In [1]:
import io
import zipfile
import urllib.request

import pandas as pd
from torch.utils.data import Dataset


try:
    bank_data = pd.read_csv("bank-full.csv", sep=";")
except FileNotFoundError:
    url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    with urllib.request.urlopen(url) as response:
        archive_data = response.read()

    with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
        csv_files = [
            name for name in archive.namelist()
            if name.endswith("bank-full.csv")
        ]

        if csv_files:
            with archive.open(csv_files[0]) as file:
                bank_data = pd.read_csv(file, sep=";")
        else:
            bank_zip = next(
                name for name in archive.namelist()
                if name.endswith("bank.zip")
            )
            with archive.open(bank_zip) as nested_file:
                with zipfile.ZipFile(io.BytesIO(nested_file.read())) as nested_archive:
                    bank_csv = next(
                        name for name in nested_archive.namelist()
                        if name.endswith("bank-full.csv")
                    )
                    with nested_archive.open(bank_csv) as file:
                        bank_data = pd.read_csv(file, sep=";")


class BankDatasetBase(Dataset):
    def __init__(self, data: pd.DataFrame) -> None:
        self.X = data.drop(columns="y")
        self.y = data["y"]

    def __getitem__(self, idx: int) -> tuple:
        return self.X.iloc[idx], self.y.iloc[idx]

    def __len__(self) -> int:
        return len(self.X)


bank_dataset_base = BankDatasetBase(bank_data)

print(f"Размер датасета: {len(bank_dataset_base)}")
x, y = bank_dataset_base[0]
print("Признаки первого объекта:")
print(x)
print(f"\nМетка первого объекта: {y}")


Размер датасета: 45211
Признаки первого объекта:
age                  58
job          management
marital         married
education      tertiary
default              no
balance            2143
housing             yes
loan                 no
contact         unknown
day                   5
month               may
duration            261
campaign              1
pdays                -1
previous              0
poutcome        unknown
Name: 0, dtype: object

Метка первого объекта: no


<p class="task" id="2"></p>

2\. Опишите класс `BankDataset`. Решение должно удовлетворять всем критериям из предыдущего задания, а также:
* при создании объекта в конструктор может быть передан необязательные аргументы `transform` и `target_transform`;
* если аргумент `transform` был передан, то при получении `i`-го элемента, нужно вызвать `transform(x)` и вернуть полученный результат.
* если аргумент `target_transform` был передан, то при получении `i`-го элемента, нужно вызвать `target_transform(y)` и вернуть полученный результат.

Создайте объект класса `BankDataset` и продемонстрируйте работоспособность (без передачи `target_transform` и `transform`).

- [ ] Проверено на семинаре

In [2]:
from typing import Callable


class BankDataset(Dataset):
    def __init__(
            self, 
            data: pd.DataFrame, 
            transform: Callable | None = None,
            target_transform: Callable | None = None
    ) -> None:
        self.X = data.drop(columns="y")
        self.y = data["y"]
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, idx: int) -> tuple:
        # x - набор признаков из idx-й строки
        # y - метка из idx-й строки
        x = self.X.iloc[idx]
        y = self.y.iloc[idx]

        if self.transform is not None:
            x = self.transform(x)
        if self.target_transform is not None:
            y = self.target_transform(y)

        return x, y

    def __len__(self) -> int:
        return len(self.X)


bank_dataset = BankDataset(bank_data)

print(f"Размер датасета: {len(bank_dataset)}")
x, y = bank_dataset[0]
print("Тип признаков:", type(x))
print("Тип метки:", type(y))
print("Метка:", y)


Размер датасета: 45211
Тип признаков: <class 'pandas.core.series.Series'>
Тип метки: <class 'str'>
Метка: no


<p class="task" id="3"></p>

3\. Опишите класс `OrdinalEncoderTransform`. Решение должно удовлетворять следующим критериям:

* при создании объекта в конструктор передаются названия нечисловых столбцов в датасете
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (признаки) и возвращает набор признаков, в котором нечисловые характеристики закодированы целыми числами;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объект класса `OrdinalEncoderTransform`.

- [ ] Проверено на семинаре

In [3]:
class Transform:
    pass


In [4]:
import pandas as pd


class OrdinalEncoderTransform(Transform):
    def __init__(self, category_columns: list[str]) -> None:
        self.category_columns = category_columns
        self.categories = {
            column: {}
            for column in category_columns
        }

    def __call__(self, x: pd.Series) -> pd.Series:
        x = x.copy()

        for column in self.category_columns:
            value = x[column]
            mapping = self.categories[column]

            if value not in mapping:
                mapping[value] = len(mapping)

            x[column] = mapping[value]

        return x


category_columns = (
    bank_data
    .drop(columns="y")
    .select_dtypes(include="object")
    .columns
    .tolist()
)

ordinal_encoder = OrdinalEncoderTransform(category_columns)
bank_dataset_encoded = BankDataset(
    bank_data,
    transform=ordinal_encoder
)

x0, y0 = bank_dataset_encoded[0]
x1, y1 = bank_dataset_encoded[1]

print("Закодированные признаки первого объекта:")
print(x0)
print("\nЗакодированные признаки второго объекта:")
print(x1)
print("\nСостояние кодировщика для столбца job:")
print(ordinal_encoder.categories["job"])


Закодированные признаки первого объекта:
age            58
job             0
marital         0
education       0
default         0
balance      2143
housing         0
loan            0
contact         0
day             5
month           0
duration      261
campaign        1
pdays          -1
previous        0
poutcome        0
Name: 0, dtype: object

Закодированные признаки второго объекта:
age           44
job            1
marital        1
education      1
default        0
balance       29
housing        0
loan           0
contact        0
day            5
month          0
duration     151
campaign       1
pdays         -1
previous       0
poutcome       0
Name: 1, dtype: object

Состояние кодировщика для столбца job:
{'management': 0, 'technician': 1}


<p class="task" id="4"></p>

4\. Опишите класс `LabelEncoderTransform`. Решение должно удовлетворять следующим критериям:

* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (строку) и возвращает целое число, соответствующее этой строке;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объекта в качестве аргумента `target_transform` объект класса `LabelEncoderTransform`.

- [ ] Проверено на семинаре

In [5]:
class LabelEncoderTransform(Transform):
    def __init__(self) -> None:
        self.classes = {}

    def __call__(self, x: str) -> int:
        if x not in self.classes:
            self.classes[x] = len(self.classes)

        return self.classes[x]


label_encoder = LabelEncoderTransform()
bank_dataset_labeled = BankDataset(
    bank_data,
    target_transform=label_encoder
)

_, y0 = bank_dataset_labeled[0]

other_idx = next(
    i for i, value in enumerate(bank_data["y"])
    if value != bank_data["y"].iloc[0]
)
_, y1 = bank_dataset_labeled[other_idx]

print("Первая закодированная метка:", y0)
print("Метка другого класса:", y1)
print("Состояние кодировщика:", label_encoder.classes)


Первая закодированная метка: 0
Метка другого класса: 1
Состояние кодировщика: {'no': 0, 'yes': 1}


<p class="task" id="5"></p>

5\. Опишите класс `ToTensor`.  Решение должно удовлетворять следующим критериям:
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает на вход серию или фрейм и возвращает тензор.

Опишите класс `Compose`.  Решение должно удовлетворять следующим критериям:
* при создании объекта в конструктор передается список объектов `transforms`, каждый из которых имеет метод `__call__(x, y)`;
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает имеет параметра (признаки и класс в числовом виде) и и возвращает кортеж, полученный путем последовательного вызова объектов из `transforms`.

Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании преобразования `Compose` список из объектов LabelEncoderTransform и ToTensor.

- [ ] Проверено на семинаре

In [6]:
import torch as th
from typing import Any


class ToTensor(Transform):
    def __call__(self, X: pd.Series | pd.DataFrame | int) -> th.Tensor:
        if isinstance(X, (pd.Series, pd.DataFrame)):
            return th.tensor(
                X.to_numpy(dtype="float32"),
                dtype=th.float32
            )

        if isinstance(X, int):
            return th.tensor(X, dtype=th.long)

        return th.as_tensor(X)


class Compose(Transform):
    def __init__(self, transforms: list[Transform]) -> None:
        self.transforms = transforms

    def __call__(self, X: Any) -> Any:
        for transform in self.transforms:
            X = transform(X)

        return X


feature_transform = Compose([
    OrdinalEncoderTransform(category_columns),
    ToTensor()
])

target_transform = Compose([
    LabelEncoderTransform(),
    ToTensor()
])

bank_dataset_tensor = BankDataset(
    bank_data,
    transform=feature_transform,
    target_transform=target_transform
)

x, y = bank_dataset_tensor[0]

print("Признаки:")
print(x)
print("Размерность признаков:", x.shape)
print("Тип признаков:", x.dtype)
print("\nМетка:", y)
print("Тип метки:", y.dtype)


Признаки:
tensor([ 5.8000e+01,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         2.1430e+03,  0.0000e+00,  0.0000e+00,  0.0000e+00,  5.0000e+00,
         0.0000e+00,  2.6100e+02,  1.0000e+00, -1.0000e+00,  0.0000e+00,
         0.0000e+00])
Размерность признаков: torch.Size([16])
Тип признаков: torch.float32

Метка: tensor(0)
Тип метки: torch.int64


<p class="task" id="6"></p>

6\. Разделите датасет из предыдущего задания на обучающую и тестовую выборку в соотношении 75% на 25%. Создайте объект `DataLoader` для получения пакетов размера 64, полученных из перемешанного обучающего датасета. Кастомизируйте `DataLoader` таким образом, чтобы пакет признаков был представлен в виде трехмерного тензора размера 64x2x8 (разделите 16 признаков на два тензора по 8). Получите один пакет и выведите на экран размерность тензоров пакета. 

- [ ] Проверено на семинаре

In [7]:
from torch.utils.data import DataLoader, random_split


train_size = int(0.75 * len(bank_dataset_tensor))
test_size = len(bank_dataset_tensor) - train_size

train_dataset, test_dataset = random_split(
    bank_dataset_tensor,
    [train_size, test_size],
    generator=th.Generator().manual_seed(42)
)


def collate_fn(batch: list[tuple]) -> tuple[th.Tensor, th.Tensor]:
    X, y = zip(*batch)

    X = th.stack(X)
    y = th.stack(y)

    X = X.reshape(X.shape[0], 2, 8)

    return X, y


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

X_batch, y_batch = next(iter(train_loader))

print("Размер обучающей выборки:", len(train_dataset))
print("Размер тестовой выборки:", len(test_dataset))
print("Размерность пакета признаков:", X_batch.shape)
print("Размерность пакета меток:", y_batch.shape)


Размер обучающей выборки: 33908
Размер тестовой выборки: 11303
Размерность пакета признаков: torch.Size([64, 2, 8])
Размерность пакета меток: torch.Size([64])


## Обратная связь
- [ ] Почта: d83073452@yandex.ru
- [ ] Gmail: 3073452@gmail.com
- [ ] Telegram: @sleepingzzzZ